# L19: 文档分割与嵌入

> RAG 系统的基础：如何将文档转化为可检索的向量

In [ ]:
# === 环境设置 ===
import sys
import os
import json
import asyncio
from pathlib import Path
from typing import List, Dict, Any, Optional

# 自动找到项目根目录
current_path = Path(os.getcwd())
project_path = None

for path in [current_path] + list(current_path.parents):
    if (path / "backend").exists():
        project_path = str(path)
        break

if project_path and project_path not in sys.path:
    sys.path.insert(0, project_path)

# 加载 .env
env_file = Path(project_path) / ".env" if project_path else None
if env_file and env_file.exists():
    with open(env_file, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                key, value = line.split("=", 1)
                os.environ[key.strip()] = value.strip()

print(f"项目路径: {project_path}")
print(f"Python 版本: {sys.version.split()[0]}")

from backend.rag.embeddings import EmbeddingModel
print("✓ 模块导入成功")

## 19.1 什么是 RAG？

### RAG = Retrieval-Augmented Generation

**RAG** = 检索增强生成，让 LLM 能够使用外部知识库回答问题。

```
传统 LLM:
  问题 → LLM → 回答 (仅依赖训练数据)
  
RAG 系统:
  问题 → 检索相关文档 → LLM 结合文档 → 回答
```

### RAG 的优势

| 优势 | 说明 |
|------|------|
| **知识更新** | 无需重新训练模型 |
| **准确回答** | 基于实际文档，减少幻觉 |
| **可解释性** | 可以追溯答案来源 |
| **成本低** | 比 fine-tuning 便宜 |
| **隐私保护** | 敏感数据不进入模型 |

## 19.2 文档分割

### 为什么需要分割？

```
问题: 直接将整个文档 Embedding
  - 文档太长超过模型限制
  - 单个向量无法捕捉细节
  - 检索结果不够精确

解决: 将文档分成小块
  - 每块语义完整
  - 块之间有重叠保持连贯
  - 检索更精确
```

### 分割策略对比

In [ ]:
from typing import List

class TextSplitter:
    """
    文本分割器基类
    """
    
    def split(self, text: str) -> List[str]:
        raise NotImplementedError

class CharacterSplitter(TextSplitter):
    """
    按字符数分割
    """
    
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
    
    def split(self, text: str) -> List[str]:
        """
        按字符数分割文本
        """
        chunks = []
        start = 0
        
        while start < len(text):
            end = start + self.chunk_size
            chunk = text[start:end]
            chunks.append(chunk)
            start = end - self.chunk_overlap
        
        return chunks

class SentenceSplitter(TextSplitter):
    """
    按句子分割（保持语义完整）
    """
    
    def __init__(self, chunk_size: int = 500, chunk_overlap: int = 50):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
    
    def split(self, text: str) -> List[str]:
        """
        按句子分割文本
        """
        import re
        
        # 分割句子
        sentences = re.split(r'([。！？.!?\n])', text)
        
        # 重组句子（保留标点）
        full_sentences = []
        for i in range(0, len(sentences) - 1, 2):
            sentence = sentences[i] + (sentences[i + 1] if i + 1 < len(sentences) else '')
            if sentence.strip():
                full_sentences.append(sentence.strip())
        
        # 组合成 chunk
        chunks = []
        current_chunk = ""
        
        for sentence in full_sentences:
            if len(current_chunk) + len(sentence) <= self.chunk_size:
                current_chunk += sentence
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                current_chunk = sentence
        
        if current_chunk:
            chunks.append(current_chunk.strip())
        
        return chunks

# 演示分割
def demo_splitting():
    # 示例文本
    sample_text = """
    Python 是一种高级编程语言，由 Guido van Rossum 于 1991 年创建。
    它以简洁、易读的语法著称，广泛应用于 Web 开发、数据科学、人工智能等领域。
    Python 的设计哲学强调代码的可读性和简洁的语法。
    Python 拥有庞大的标准库和第三方生态系统。
    Django 和 Flask 是流行的 Web 框架。
    NumPy 和 Pandas 是数据科学的核心库。
    TensorFlow 和 PyTorch 是深度学习的框架。
    """
    
    print("=== 文档分割演示 ===\n")
    print(f"原文长度: {len(sample_text)} 字符\n")
    
    # 字符分割
    char_splitter = CharacterSplitter(chunk_size=100, chunk_overlap=20)
    char_chunks = char_splitter.split(sample_text.strip())
    
    print("字符分割:")
    for i, chunk in enumerate(char_chunks, 1):
        print(f"  Chunk {i}: {chunk[:60]}... ({len(chunk)} 字符)")
    
    print("\n" + "="*50 + "\n")
    
    # 句子分割
    sent_splitter = SentenceSplitter(chunk_size=100, chunk_overlap=20)
    sent_chunks = sent_splitter.split(sample_text.strip())
    
    print("句子分割:")
    for i, chunk in enumerate(sent_chunks, 1):
        print(f"  Chunk {i}: {chunk[:60]}... ({len(chunk)} 字符)")

demo_splitting()

### 分割最佳实践

| 建议 | 说明 |
|------|------|
| **Chunk 大小** | 500-1000 字符，根据内容调整 |
| **Overlap** | 10-20%，保持上下文连贯 |
| **边界** | 优先在句子、段落边界分割 |
| **语义完整** | 每个 chunk 应该有独立意义 |
| **特殊处理** | 代码、表格需要专门处理 |

## 19.3 文本嵌入 (Embeddings)

### Embedding 过程

In [ ]:
# 查看项目中的 Embedding 实现
import inspect
from backend.rag.embeddings import EmbeddingModel

print("=== EmbeddingModel 源码 ===\n")
print(inspect.getsource(EmbeddingModel))

### Embedding 的维度和意义

In [ ]:
import numpy as np
from typing import List

def visualize_embedding(text: str, embedding: List[float], dim: int = 10):
    """
    可视化 Embedding 向量（前 dim 维）
    """
    print(f"文本: {text}")
    print(f"向量维度: {len(embedding)}")
    print(f"前 {dim} 维: {np.round(embedding[:dim], 4)}")
    print(f"向量范数: {np.linalg.norm(embedding):.4f}")

# 模拟 Embedding（实际需要调用模型）
def mock_embedding(text: str, dim: int = 384) -> List[float]:
    """
    模拟生成 Embedding（仅用于演示）
    """
    np.random.seed(hash(text) % 10000)  # 使用文本作为种子
    vec = np.random.randn(dim)
    # 归一化
    vec = vec / np.linalg.norm(vec)
    return vec.tolist()

# 演示
def demo_embeddings():
    print("=== Embedding 演示 ===\n")
    
    texts = [
        "猫是可爱的宠物",
        "狗是人类最好的朋友",
        "Python 是编程语言",
    ]
    
    embeddings = [mock_embedding(t) for t in texts]
    
    for text, emb in zip(texts, embeddings):
        visualize_embedding(text, emb)
        print()
    
    # 计算相似度
    print("相似度矩阵:")
    print("          " + "  ".join([f"[{i}]" for i in range(len(texts))]))
    
    for i, text_i in enumerate(texts):
        row = f"[{i}] {text_i[:10]:10}"
        for j, text_j in enumerate(texts):
            sim = np.dot(embeddings[i], embeddings[j])
            row += f" {sim:5.2f}"
        print(row)

demo_embeddings()

## 19.4 完整的文档处理流程

In [ ]:
class DocumentProcessor:
    """
    文档处理器 - 完整的分割和嵌入流程
    """
    
    def __init__(
        self,
        splitter: TextSplitter = None,
        embedding_model = None
    ):
        self.splitter = splitter or SentenceSplitter()
        self.embedding_model = embedding_model
    
    async def process(
        self,
        text: str,
        metadata: Dict[str, Any] = None
    ) -> List[Dict[str, Any]]:
        """
        处理文档：分割 → 嵌入
        """
        # 1. 分割
        chunks = self.splitter.split(text)
        
        results = []
        
        # 2. 嵌入每个 chunk
        for i, chunk in enumerate(chunks):
            if self.embedding_model:
                embedding = await self.embedding_model.embed(chunk)
            else:
                embedding = mock_embedding(chunk)
            
            chunk_metadata = {
                "chunk_index": i,
                "total_chunks": len(chunks),
                "chunk_length": len(chunk)
            }
            if metadata:
                chunk_metadata.update(metadata)
            
            results.append({
                "content": chunk,
                "embedding": embedding,
                "metadata": chunk_metadata
            })
        
        return results

# 演示完整流程
async def demo_processing():
    print("=== 文档处理完整流程 ===\n")
    
    processor = DocumentProcessor(
        splitter=SentenceSplitter(chunk_size=150),
        embedding_model=None  # 使用 mock
    )
    
    doc = """
    Python 是一种高级编程语言，由 Guido van Rossum 创建。
    它以简洁的语法著称，广泛应用于 Web 开发和数据分析。
    Django 和 Flask 是流行的 Web 框架。
    NumPy 和 Pandas 是数据科学的核心库。
    """
    
    results = await processor.process(
        doc.strip(),
        metadata={"source": "python_intro.txt", "category": "programming"}
    )
    
    print(f"原文档被分成 {len(results)} 个 chunk:\n")
    
    for r in results:
        print(f"Chunk {r['metadata']['chunk_index'] + 1}/{r['metadata']['total_chunks']}:")
        print(f"  内容: {r['content'][:50]}...")
        print(f"  长度: {r['metadata']['chunk_length']} 字符")
        print(f"  向量: {r['embedding'][:5]}... (共 {len(r['embedding'])} 维)")
        print(f"  元数据: {r['metadata']}")
        print()

await demo_processing()

## 19.5 常见面试问题

**Q: Chunk 大小如何选择？**
- A:
  - **太小**（<200）：语义不完整，检索结果碎片化
  - **太大**（>1500）：检索结果包含太多无关信息
  - **推荐**：500-1000 字符，根据具体内容调整
  - **公式**：chunk_size ≈ 2-3 个典型句子

**Q: 为什么需要 Overlap？**
- A:
  1. 保持上下文连贯性
  2. 防止关键信息被分割
  3. 提高检索召回率
  4. 推荐值：10-20% 的 chunk_size

**Q: 如何处理代码文档？**
- A:
  1. 按函数/类分割，保持代码完整性
  2. 保留注释和文档字符串
  3. 考虑语法高亮信息
  4. 可以使用专门的代码分割器

**Q: Embedding 模型如何选择？**
- A:
  | 场景 | 推荐模型 |
  |------|----------|
  | 中文为主 | bge-large-zh, m3e-base |
  | 英文为主 | text-embedding-3-small |
  | 多语言 | multilingual-e5 |
  | 本地部署 | all-MiniLM-L6-v2 |
  | 高质量 | OpenAI text-embedding-3-large |

**Q: 如何评估 Embedding 质量？**
- A:
  1. **检索准确率**：相关文档的检索排名
  2. **聚类效果**：相似文档的距离
  3. **下游任务**：RAG 的问答质量
  4. **基准测试**：MTEB 排行榜

---

## 总结

本课程学习了文档分割与嵌入的核心知识：

| 知识点 | 说明 |
|--------|------|
| **RAG 概念** | 检索增强生成 |
| **文档分割** | 将长文档分成有意义的块 |
| **分割策略** | 字符分割、句子分割、语义分割 |
| **参数选择** | chunk_size、chunk_overlap |
| **文本嵌入** | 将文本转化为向量表示 |
| **Embedding** | 捕捉语义信息的数值向量 |

**下一步**: L20 将学习向量检索技术！